# D-algebraic functions expansion

In [1]:
import sys
sys.path.insert(0, "..") # dalgebra is here
from dalgebra import *
from dalgebra.pseries.laurent import *

%display latex

I want to think about what to do to detect possible orders for Laurent series expansions around 0 of solutions of D-algebraic equations. 

I am not fuly sure on all the steps but I have an intuition on how to proceed.

In order to work properly, I am going to need at least three examples:
* A linear example: things are easy here and a polynomial (indicial polynomial) can be computed.
* A purely D-algebraic case:
  - With guaranteed highest order monomial with its highest derivative
  - With highest order monomial without its highest derivative.
 
Then, it remains to check how to compute the expansion of the Laurent series solutions and how can we actually compute which inicial conditions are needed.

## Generating the power series solution

Given an equation, we may want to look when it has a "low order" solution. This requires to set up a given ansatz and then extend the possible solution. After that, we need to check the relations between the initial conditions to have an actual solution to the equation.

In [2]:
def generic_solution(equation, gen):
    R = equation.parent()
    C = R.constant_ring()
    order = equation.order(gen)

    nC = C.add_constants(*[f"a_{i}" for i in range(order)])
    a = nC.constant_ring().gens()
    return equation.power_series_solution(gen, {i: a[i] for i in range(order)})

## The linear case

In this case, we have an equation of the shape:
$$L = \alpha_0 u+ \alpha_1 u' + \ldots + \alpha_n u^{(n)}.$$
We assume the elements $\alpha_i \in C[[x]]$, so thei can only have a positive order.

Hence, each term will have order $d - i + ord(\alpha_i)$. This makes comparisons quite trivial and independent of $d$.

In [3]:
C = DifferentialRing(QQ)
R = DifferentialRing(QQ[x], [1])
R.set_constant(C)
F = R.fraction_field()
x, = F.gens()
DO.<u> = DifferentialPolynomialRing(F)
LS.<t> = LaurentSeries(C)
mor = F.laurent_morphism({'x': t}, set_default=True)

### 1.1. A monic case

$$L = u^{(5)} - 1/(1-x) u^{(3)} + 2xu'' - u$$

In [4]:
L = u[5] - (1/(1-x))*u[3] + 2*x*u[2] - u[0]
L

-u_0 + (2*x)*u_2 + ((-1)/(-x + 1))*u_3 + u_5

In [5]:
L.indicial_equation(u, "k", laurent_morph=mor)

((), (), None)

Since we have zero candidates, then we conclude that the order of the series must be greater or equal than 0 and smaller than $5$.

We can check now a generic solution for relations among the initial conditions to be solutions of our equation:

In [6]:
sol = generic_solution(L, u)
mor_sol = L.parent().laurent_morphism(imgs={'u': sol}, constant=sol.parent().constant_ring())
L_eval = mor_sol(L)
L_eval.is_zero()

20

Since we get zero in the evaluation, we can conclude that up to order 20, we have no condition, hence, any possibility for the constants in the generic solution provide a solution to the equation.

This makes sense, because we are considering a linear differential equation, whose solutions are a C-vector space and the initial conditions guarantee a linear independence. 

### 1.2. The special case: Bessel differential equation

In [7]:
L = x^2*u[2] + x*u[1] + (x^2-25)*u[0]
#orders = [goal_order(m,c,k) for m,c in (zip(L.monomials(u), L.coefficients(u)))]
#candidates = []
#for i in range(len(orders)):
#    for j in range(i+1, len(orders)):
#         to_root = orders[i][0] - orders[j][0]
#         if to_root == 0:
#             candidates.append(orders[i][0])
#         else:
#             candidates.append((ord_i - ord_j).roots())
# candidates

In [8]:
L.indicial_equation(u, "k", laurent_morph=mor)

((((-Infinity, +Infinity), [5, -5]),), (), None)

In this case, we have an equation so homogeneous that any function with order different than $(0,1)$ may have cancellations. So these candidates are not good enough. Let us dive a bit deeper. If we assume a good order (i.e., different than $0,1$) then we have that 
$$[x^{k}] L\cdot u(x) = k(k-1)u_{k} + ku_{k} - 25u_k = (k^2 - k + k - 25) u_k = (k^2 - 25)u_k.$$
Since we assume the order of $u$ is $k$, then we know that $u_k \neq 0$. Hence we obtain $k \in \{\pm5\}$. These are the integer roots of the actual inidicial polynomial. This case is also quite interesting since extending the initial values is not as simple as it may seem using the classical approach of computing $u''$ from the equation and derivating here.

In this case, we use the following approach:
$$[x^p] L\cdot u(x) = [x^p] x^2u'' + [x^p] xu' + [x^p](x^2-25)u = [x^{p-2}] u'' + [x^{p-1}] u' + [x^{p-2}]u - 25 [x^p]u = p(p-1)u_p + pu_p + u_{p-2} - 25u_p = (p^2 - 25)u_p + u_{p-2},$$
so we can get the value:
$$u_{p} = \frac{u_{p-2}}{(25-p^2)},$$
which is a very simple formula tha works for any $p \notin\{\pm5\}$.

We could easily get this formulation due to the nature of the equation $L$: is is a D-finite equation (linear with polynomial coefficients) so it is known that the sequence of any solution is $P$-finite (or define with a linear recurrence with polynomial coefficients).

In [9]:
def _bessel_coeff(k,n):
    if k < n:
        return 0
    elif k == n:
        return 1/3840
    else:
        return _bessel_coeff(k-2,n)/(n^2 - k^2)
sol = LS.element_class(LS, coefficient_map=lambda k : _bessel_coeff(k,5), order=5)

In [10]:
[sol[i]*factorial(i) for i in range(5,10)]

[1/32, 0, -7/128, 0, 9/128]

In [11]:
f = Bessel(5)(SR('t'))

In [12]:
[f.derivative(i)(t=0) for i in range(20)]

[0,
 0,
 0,
 0,
 0,
 1/32,
 0,
 -7/128,
 0,
 9/128,
 0,
 -165/2048,
 0,
 715/8192,
 0,
 -3003/32768,
 0,
 1547/16384,
 0,
 -12597/131072]

In [13]:
T =f.variables()[0]

## The non-linear case

Non-linear differential equations are quite common in the context of this repository. A classic example is the $\wp$-Weierstrass function:
$$\wp'^2 = 4\wp^3 + g_2\wp + g_3.$$
Let us check how the indicail method works on this equation:

In [4]:
DO_C = DO.add_constants("g_2", "g_3")
u = DO_C.gen("u")
C_C = DO_C.constant_ring()
F_C = DO_C.base()
x,g_2,g_3 = F_C.gens()
LS_C = LS.add_constants("g_2", "g_3")
t_C = LS_C.gen()
mor = F_C.laurent_morphism({'x': t_C}, set_default=True)

In [5]:
L = u[1]^2 - 4*u[0]^3 - g_2*u[0] - g_3
L

-g_3 - g_2*u_0 - 4*u_0^3 + u_1^2

In [6]:
L.indicial_equation(u, laurent_morph=mor)

ValueError: No default Laurent morphism has been set

In [8]:
L(g_3=0).indicial_equation(u, laurent_morph=mor)

((), ((0, 'Small'), (2, -g_2*u_0 + 4*u_0^2)), None)

In [9]:
Lp = L.derivative() // u[1]

In [10]:
Lp.indicial_equation(u, laurent_morph=mor)

((), ((-2, -12*u_0^2 + 12*u_0), (0, 'Small'), (2, -g_2 + 4*u_0)), None)

In [109]:
DO_p = Lp.parent().add_constants("a_0", "a_1","aux")
u_p = DO_p.gen("u")
C_p = DO_p.constant_ring()
F_p = DO_p.base()
x_p,g_2_p,g_3_p,a_0,a_1,aux = F_p.gens()
LS_p = LS_C.add_constants("a_0", "a_1", "aux")
t_p = LS_p.gen()
mor = F_p.laurent_morphism({'x': t_p}, set_default=True)

In [110]:
gen_sol=Lp.power_series_solution(Lp.parent().gen("u"), {0: a_0, 1: a_1})

In [111]:
I_gens = [((1-aux*a_0)*(1-aux*a_1)).numerator().wrapped] # at least one initial condition is not zero
I = [ideal(ideal(I_gens).groebner_basis()).elimination_ideal(aux.numerator().wrapped)]
I

[Ideal (0) of Multivariate Polynomial Ring in x, g_2, g_3, a_0, a_1, aux over Rational Field]

In [114]:
I_gens.append(gen_sol[len(I_gens)-1].numerator().wrapped)
I.append(ideal(ideal(I_gens).groebner_basis()).elimination_ideal(aux.numerator().wrapped))
I

[Ideal (0) of Multivariate Polynomial Ring in x, g_2, g_3, a_0, a_1, aux over Rational Field,
 Ideal (a_0) of Multivariate Polynomial Ring in x, g_2, g_3, a_0, a_1, aux over Rational Field,
 Ideal (1) of Multivariate Polynomial Ring in x, g_2, g_3, a_0, a_1, aux over Rational Field,
 Ideal (1) of Multivariate Polynomial Ring in x, g_2, g_3, a_0, a_1, aux over Rational Field]

In [72]:
PolynomialRing?

Signature:      PolynomialRing(base_ring, *args, **kwds)
Docstring:     
   Return the globally unique univariate or multivariate polynomial
   ring with given properties and variable name or names.

   There are many ways to specify the variables for the polynomial
   ring:

   1. "PolynomialRing(base_ring, name, ...)"

   2. "PolynomialRing(base_ring, names, ...)"

   3. "PolynomialRing(base_ring, n, names, ...)"

   4. "PolynomialRing(base_ring, n, ..., var_array=var_array, ...)"

   The "..." at the end of these commands stands for additional
   keywords, like "sparse" or "order".

   INPUT:

   * "base_ring" -- a ring

   * "n" -- integer

   * "name" -- string

   * "names" -- list or tuple of names (strings), or a comma separated
     string

   * "var_array" -- list or tuple of names, or a comma separated
     string

   * "sparse" -- boolean; whether or not elements are sparse. The
     default is a dense representation ("sparse=False") for univariate
     rings and a sparse r

In [22]:
gen_sol.derivative(times=2)

2*a_0^2*1 + 12*a_0*a_1*t + (24*a_0^3 + 12*a_1^2)*t^2 + 120*a_0^2*a_1*t^3 + (150*a_0^4 + 180*a_0*a_1^2)*t^4 + (840*a_0^3*a_1 + 84*a_1^3)*t^5 + (784*a_0^5 + 1680*a_0^2*a_1^2)*t^6 + (5040*a_0^4*a_1 + 1440*a_0*a_1^3)*t^7 + (3780*a_0^6 + 12600*a_0^3*a_1^2 + 450*a_1^4)*t^8 + (27720*a_0^5*a_1 + 15400*a_0^2*a_1^3)*t^9 + (17424*a_0^7 + 83160*a_0^4*a_1^2 + 9240*a_0*a_1^4)*t^10 + (144144*a_0^6*a_1 + 131040*a_0^3*a_1^3 + 2184*a_1^5)*t^11 + (78078*a_0^8 + 504504*a_0^5*a_1^2 + 114660*a_0^2*a_1^4)*t^12 + (720720*a_0^7*a_1 + 970200*a_0^4*a_1^3 + 52920*a_0*a_1^5)*t^13 + (343200*a_0^9 + 2882880*a_0^6*a_1^2 + 1108800*a_0^3*a_1^4 + 10080*a_1^6)*t^14 + (3500640*a_0^8*a_1 + 6534528*a_0^5*a_1^3 + 753984*a_0^2*a_1^5)*t^15 + (1487772*a_0^10 + 15752880*a_0^7*a_1^2 + 9189180*a_0^4*a_1^4 + 282744*a_0*a_1^6)*t^16 + (16628040*a_0^9*a_1 + 41081040*a_0^6*a_1^3 + 8216208*a_0^3*a_1^5 + 45144*a_1^7)*t^17 + (6382480*a_0^11 + 83140200*a_0^8*a_1^2 + 68468400*a_0^5*a_1^4 + 4564560*a_0^2*a_1^6)*t^18 + (77597520*a_0^10*a_1 + 245044800*a_0^7*a_1^3 + 75675600*a_0^4*a_1^5 + 1441440*a_0*a_1^7)*t^19 + (27159132*a_0^12 + 426786360*a_0^9*a_1^2 + 471711240*a_0^6*a_1^4 + 55495440*a_0^3*a_1^6 + 198198*a_1^8)*t^20 + O(t^21)

In [27]:
sol_eval = -g_2_p  - 12*gen_sol*gen_sol + 2*gen_sol.derivative(times=2)

In [42]:
I_gens = [sol_eval[0].numerator().wrapped]

In [43]:
ideal(I_gens).groebner_basis()

[a_0^2 + 1/8*g_2]

In [57]:
I_gens.append(sol_eval[len(I_gens)].numerator().wrapped)
ideal(I_gens).groebner_basis()

[a_0*a_1^2, a_1^3, g_2^2, g_2*a_0 - 4*a_1^2, a_0^2 + 1/8*g_2, g_2*a_1]

In [51]:
I_gens

[-8*a_0^2 - g_2,
 0,
 24*a_0^3 + 12*a_1^2,
 168*a_0^2*a_1,
 240*a_0^4 + 288*a_0*a_1^2,
 1440*a_0^3*a_1 + 144*a_1^3,
 1400*a_0^5 + 3000*a_0^2*a_1^2,
 9240*a_0^4*a_1 + 2640*a_0*a_1^3]

In [37]:
sol_eval

-(8*a_0^2 + g_2)*1 + (24*a_0^3 + 12*a_1^2)*t^2 + 168*a_0^2*a_1*t^3 + (240*a_0^4 + 288*a_0*a_1^2)*t^4 + (1440*a_0^3*a_1 + 144*a_1^3)*t^5 + (1400*a_0^5 + 3000*a_0^2*a_1^2)*t^6 + (9240*a_0^4*a_1 + 2640*a_0*a_1^3)*t^7 + (7056*a_0^6 + 23520*a_0^3*a_1^2 + 840*a_1^4)*t^8 + (52416*a_0^5*a_1 + 29120*a_0^2*a_1^3)*t^9 + (33264*a_0^7 + 158760*a_0^4*a_1^2 + 17640*a_0*a_1^4)*t^10 + (277200*a_0^6*a_1 + 252000*a_0^3*a_1^3 + 4200*a_1^5)*t^11 + (151008*a_0^8 + 975744*a_0^5*a_1^2 + 221760*a_0^2*a_1^4)*t^12 + (1400256*a_0^7*a_1 + 1884960*a_0^4*a_1^3 + 102816*a_0*a_1^5)*t^13 + (669240*a_0^9 + 5621616*a_0^6*a_1^2 + 2162160*a_0^3*a_1^4 + 19656*a_1^6)*t^14 + (6846840*a_0^8*a_1 + 12780768*a_0^5*a_1^3 + 1474704*a_0^2*a_1^5)*t^15 + (2917200*a_0^10 + 30888000*a_0^7*a_1^2 + 18018000*a_0^4*a_1^4 + 554400*a_0*a_1^6)*t^16 + (32672640*a_0^9*a_1 + 80720640*a_0^6*a_1^3 + 16144128*a_0^3*a_1^5 + 88704*a_1^7)*t^17 + (12563408*a_0^11 + 163654920*a_0^8*a_1^2 + 134774640*a_0^5*a_1^4 + 8984976*a_0^2*a_1^6)*t^18 + (152977968*a_0^10*a_1 + 483088320*a_0^7*a_1^3 + 149189040*a_0^4*a_1^5 + 2841696*a_0*a_1^7)*t^19 + (53612832*a_0^12 + 842487360*a_0^9*a_1^2 + 931170240*a_0^6*a_1^4 + 109549440*a_0^3*a_1^6 + 391248*a_1^8)*t^20 + O(t^21)

In [40]:
ideal(I_gens)

Principal ideal (1) of Fraction Field of Differential Ring [[Multivariate Polynomial Ring in g_2, g_3, a_0, a_1 over Rational Field], (0,)]

In [41]:
I_gens

[-8*a_0^2 - g_2, 0]